In [5]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils.text import extract_text
import re
from random import choice
import shutil

In [6]:
ROOT = Path("~/code/selector_examenes").expanduser()

Diccionarios de sustitución de caracteres con `str.maketrans()`

In [7]:
SUBSCRIPT = str.maketrans("0123456789+-–‒−", "₀₁₂₃₄₅₆₇₈₉₊₋₋₋₋")
SUPERSCRIPT = str.maketrans("0123456789+-–‒−", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁻⁻⁻")

In [8]:
def left_neighbor(char, chars_by_x1):
    """Carácter que termina justo donde empieza `char` (a menos de 1,5 pt) y a su misma altura."""
    x = round(char["x0"])
    candidates = [other for key in (x - 1, x, x + 1) for other in chars_by_x1.get(key, [])
                  if other is not char
                  and abs(other["x1"] - char["x0"]) < 1.5
                  and other["top"] < char["bottom"] and char["top"] < other["bottom"]]
    return min(candidates, key=lambda other: abs(other["x1"] - char["x0"]), default=None)

In [9]:
def extract_text_from_page(page):
    """Texto de la página convirtiendo los sub/superíndices en caracteres Unicode."""
    # índice de los caracteres por su borde derecho (redondeado) para encontrar vecinos rápido
    chars_by_x1 = {}
    for char in page.chars:
        chars_by_x1.setdefault(round(char["x1"]), []).append(char)

    base_of = {}       # id(índice) -> carácter normal al que acompaña (la "H" de "H₂")
    characters = []
    for char in sorted(page.chars, key=lambda char: char["x0"]):   # de izquierda a derecha
        left = left_neighbor(char, chars_by_x1)
        # si el vecino ya es un índice (el "¹" de "10⁻¹⁴"), la base es la de ese vecino
        base = base_of.get(id(left), left)
        if base is not None and char["size"] < 0.9 * base["size"]:
            # superíndice si su base queda claramente por encima de la del carácter normal
            table = SUPERSCRIPT if char["bottom"] < base["bottom"] - 2 else SUBSCRIPT
            base_of[id(char)] = base
            # se le da la altura de su base para que extract_text lo ponga en la misma línea
            char = dict(char, text=char["text"].translate(table), top=base["top"], bottom=base["bottom"])
        characters.append(char)
    # extract_text junta en una línea los caracteres cuyo top difiere como máximo y_tolerance
    # y dentro de cada línea los ordena por x0
    return extract_text(characters, y_tolerance=4)

In [10]:
def clean_page_text_2020_2024(page_text):
    """Formatea el texto del examen usando el formato de examen de 2020 - 2024"""
    text_lines = page_text.split("\n")
    # obtener índices de líneas a eliminar
    bloque_line_indexes = []
    for index, line in enumerate(text_lines):
        if "BLOQUE" in line:
            bloque_line_indexes.append(index)

    # borrar línea que empieza por BLOQUE y
    # las 3 siguientes (BLOQUE A) o
    # las 4 siguientes (BLOQUE B, C)
    line_indexes_to_remove = []
    # en cada línea que empieza por BLOQUE,
    # se marca esa línea y las 4 (ó 3) siguientes líneas
    for idx in bloque_line_indexes:
        if text_lines[idx].startswith("BLOQUE A"):
            line_indexes_to_remove.extend(list(range(idx, idx+4)))
        else:
            line_indexes_to_remove.extend(list(range(idx, idx+5)))

    # filtrar las líneas para excluir las instrucciones del bloque        
    page_text = "\n".join([text_lines[idx] for idx in range(len(text_lines))
                        if idx not in line_indexes_to_remove])

    # inserta una línea en blanco bajo cada ejercicio
    page_text = re.sub(r"(^[ABC]\d{1}\.)", r"\n\1", page_text, flags=re.MULTILINE)

    return page_text

In [11]:
def extract_exercises_from_pdf(file: Path) -> str:
    match_year = re.search(r"\d{4}", file.stem)
    if match_year:
        year = int(match_year.group(0))

    # inserto cabecera con nombre del archivo
    exam_text = f"{file.stem}\n" + "-" * len(file.stem) + "\n"

    # string donde comienzan los ejercicios
    if year <= 2019:
        start_string = "OPCIÓN"
    else:
        start_string = r"[AB]\d{1}\."

    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            page_text = extract_text_from_page(page)
            
            match = re.search(start_string, page_text)
            
            # recortar líneas iniciales hasta start_string
            if match:
                page_text = page_text[match.start():]

            # convertir a minúsculas la opción
            page_text = (page_text
                         .replace("OPCIÓN", "Opción")
                         .replace("Opción B", "\n\nOpción B")
            )

            # eliminar líneas de bloque e instrucciones
            # en exámenes 2020+
            if year >= 2020:
                page_text = clean_page_text_2020_2024(page_text)

            else:  # examen 2010 - 2019
                # inserta una línea en blanco bajo cada ejercicio
                page_text = re.sub(r"(^\d{1}\.-)", r"\n\1", page_text, flags=re.MULTILINE)
                    
            exam_text += f"{page_text}"

    return exam_text

In [42]:
def split_exercises(exam_text: str, year_period: int) -> list:
    """Divide el texto del examen en ejercicios.
    El parámetro year_period ajusta el nombre de los ejercicios
    acorde al periodo del examen:
    - 2010 - 2019: year_period = 0
    - 2020 - 2024: year_period = 1
    - 2025 - : year_period = 2
    """
    if year_period not in range(3):
        raise ValueError("Código de año de examen incorrecto.")

    # pattern to split exercises
    # depending on exercise year period
    if year_period == 0:
        pattern = r"^\d{1}\.-"  # 3.-
    elif year_period == 1:
        pattern = r"^[ABC]\d{1}\."  # B2.
    else:
        pattern = r"^\d{1}[AB]\.|PREGUNTA 5\."

    # extract exercise numbers
    exercise_numbers = re.findall(pattern, exam_text, flags=re.MULTILINE)

    # split exam_text using exercise number pattern
    exercises = re.split(pattern, exam_text, flags=re.M)

    # concatenate exercise number and exercise statement
    # exercises start from index 1
    exercises_list = [f"{ex_number}{exercise}" for ex_number, exercise in zip(exercise_numbers, exercises[1:])]

    # special exercise number for competencial exercise
    # in exams from 2024
    if year_period == 2:
        # extract last exercise ("competencial")
        competencial_exercise = exercises_list[-1]
        # split into lines
        competencial_lines = competencial_exercise.split("\n")
        # shorten first line of competencial
        competencial_lines[0] = "PREGUNTA 5.-"
        # join all lines back
        competencial_exercise = "\n".join(competencial_lines)
        # replace modified last exercise into exercises list
        exercises_list[-1] = competencial_exercise

    return exercises_list

In [ ]:
file_2013 = ROOT / "examenes_quimica/2013_quimica/2013_quimica_junio_titular.pdf"
file_2022 = ROOT / "examenes_quimica/2022_quimica/2022_quimica_ordinaria_suplente.pdf"
file_2025 = ROOT / "examenes_quimica/2026_quimica/2026_quimica_extraordinaria_titular_ex.pdf"

In [54]:
exam_text_2013 = extract_exercises_from_pdf(file_2013)
exam_text_2022 = extract_exercises_from_pdf(file_2022)
exam_text_2025 = extract_exercises_from_pdf(file_2025)

In [55]:
print(exam_text_2013)

2013_quimica_junio_titular
--------------------------
Opción A

1.- Formule o nombre los siguientes compuestos: a) Hidróxido de paladio(II) b) Ácido sulfúrico c) Ácido
2-aminopropanoico d) BeH₂ e) Ag₃AsO₄ f) CH₃CH₂CH₂OH

2.- Para los siguientes elementos Na, P, S y Cl, diga razonadamente cuál es:
a) El de menor energía de ionización.
b) El de mayor afinidad electrónica.
c) El de mayor radio atómico.

3.- Justifique la veracidad o falsedad de las siguientes afirmaciones:
a) Como el producto de solubilidad del cloruro de plata es 2,8·10⁻¹⁰, la solubilidad en agua de esta sal es
3·10⁻³ M.
b) En toda disolución saturada de hidróxido de magnesio se cumple: [OH][Mg²⁺]² = KS.
c) Todos los hidróxidos poco solubles se hacen aún más insolubles en medio básico.

4.- La fórmula molecular del azúcar común o azúcar de mesa (sacarosa) es C₁₂H₂₂O₁₁. Indique
razonadamente si 1 mol de sacarosa contiene:
a) 144 g de carbono.
b) 18 mol de átomos de carbono.
c) 5·10¹⁵ átomos de carbono.
Datos: Masas atómi

In [56]:
exercises_list_2013 = split_exercises(exam_text_2013, year_period=0)
exercises_list_2022 = split_exercises(exam_text_2022, year_period=1)
exercises_list_2025 = split_exercises(exam_text_2025, year_period=2)

In [57]:
print(exercises_list_2013[-1])

6.- A 350 K la constante de equilibrio Kc de la reacción de descomposición del bromuro de carbonilo vale
0,205: COBr₂(g) CO(g) + Br₂(g). Si en un recipiente de 3 L se introducen 3,75 mol de bromuro de
carbonilo y se calienta hasta alcanzar esa temperatura:
a) ¿Cuáles son las concentraciones de todas las especies en equilibrio?
b) ¿Cuál es el grado de disociación del bromuro de carbonilo en esas condiciones?


In [58]:
print(exercises_list_2022[9])

C2. Mediante la electrolisis de sales fundidas se pueden obtener metales puros.
a) Escriba la semireacción que tiene lugar en el cátodo y calcule la carga eléctrica necesaria para depositar 25 g de Ni a partir de NiSO₄
fundido.
b) Determine la masa atómica del Cu si, al hacer pasar una corriente de 10 A durante 45 minutos a través de CuSO₄ fundido, se depositan
8,89 g de Cu.
Datos: F= 96500 C·mol⁻¹; Masa atómica relativa: Ni= 58,7




In [59]:
print(exercises_list_2025[-1])

PREGUNTA 5.-
DESINFECCIÓN EN EL HOGAR: EL PAPEL QUÍMICO DE LA LEJÍA
En la limpieza doméstica es frecuente el uso de lejía para desinfectar superficies de cocina y baño. La lejía comercial es una disolución
acuosa de NaClO (M= 74,5 g·mol⁻¹), cuya concentración suele rondar el 5% en masa, con una densidad de 1,08 g·mL⁻¹, aunque existen
formulaciones más concentradas destinadas a usos profesionales. El ion hipoclorito actúa como agente oxidante y desinfectante. En agua,
el ion hipoclorito se hidroliza parcialmente, generando pequeñas cantidades de ácido hipocloroso (Ka= 3,5·10⁻⁸), un desinfectante incluso
más eficaz que el propio ion hipoclorito debido a su mayor capacidad para atravesar las membranas celulares de los microorganismos.
Cuando la lejía se almacena durante largos periodos, especialmente en lugares cálidos o expuestos a la luz, el NaClO se descompone
lentamente produciendo cloruro de sodio y NaClO₃. A temperatura ambiente, después de seis meses, es habitual que al menos un 8%